In [2]:
%sql
with denominator_patient_groups as (

    select
        source_version,
        cohort_id,
        patient_key,
        organism_reporting_group,

        max(
            case
                when ast_cre_status <> 'not_applicable'
                    then 1
                else 0
            end
        ) as cre_eligible,

        max(
            case
                when ast_mdr_p_aeruginosa_status <> 'not_applicable'
                    then 1
                else 0
            end
        ) as mdr_p_aeruginosa_eligible

    from workspace.micro_dev.int_reporting_culture_organism

    group by
        source_version,
        cohort_id,
        patient_key,
        organism_reporting_group
),

expected_denominators as (

    select
        source_version,
        cohort_id,
        organism_reporting_group as organism_base_name,
        'CRE' as mdro_phenotype,
        count(*) as expected_denominator_count

    from denominator_patient_groups

    where cre_eligible = 1

    group by
        source_version,
        cohort_id,
        organism_reporting_group

    union all

    select
        source_version,
        cohort_id,
        organism_reporting_group,
        'MDR Pseudomonas aeruginosa',
        count(*)

    from denominator_patient_groups

    where mdr_p_aeruginosa_eligible = 1

    group by
        source_version,
        cohort_id,
        organism_reporting_group
),

expected_numerators as (

    select
        source_version,
        cohort_id,
        organism_reporting_group as organism_base_name,

        case
            when reporting_phenotype = 'cre'
                then 'CRE'

            when reporting_phenotype = 'mdr_p_aeruginosa'
                then 'MDR Pseudomonas aeruginosa'
        end as mdro_phenotype,

        count(*) as expected_positive_count

    from workspace.micro_dev.int_first_reporting_isolate

    where reporting_phenotype in (
        'cre',
        'mdr_p_aeruginosa'
    )

    group by
        source_version,
        cohort_id,
        organism_reporting_group,
        reporting_phenotype
),

comparison as (

    select
        m.source_version,
        m.cohort_id,
        m.organism_base_name,
        m.mdro_phenotype,

        m.denominator_count,
        d.expected_denominator_count,

        m.positive_count,
        coalesce(
            n.expected_positive_count,
            0
        ) as expected_positive_count,

        m.not_detected_count,
        m.not_evaluable_count,
        m.phenotype_pct,

        case
            when m.denominator_count
                 <> d.expected_denominator_count
                then 'denominator_mismatch'

            when m.positive_count
                 <> coalesce(
                     n.expected_positive_count,
                     0
                 )
                then 'numerator_mismatch'

            else 'pass'
        end as validation_status

    from workspace.micro_dev.mart_mdro m

    left join expected_denominators d
        on m.source_version = d.source_version
       and m.cohort_id = d.cohort_id
       and m.organism_base_name = d.organism_base_name
       and m.mdro_phenotype = d.mdro_phenotype

    left join expected_numerators n
        on m.source_version = n.source_version
       and m.cohort_id = n.cohort_id
       and m.organism_base_name = n.organism_base_name
       and m.mdro_phenotype = n.mdro_phenotype

)

select
    validation_status,
    count(*) as mart_rows

from comparison

group by
    validation_status

order by
    validation_status

,validation_status,mart_rows
0,pass,31
